In [1]:
import xml.etree.ElementTree as ET
import csv

NS = {"db": "http://www.drugbank.ca"}


In [2]:
def get_text(x):
    if x is None or x.text is None:
        return None
    return x.text.strip()

def get_primary_id(drug):
    for dbid in drug.findall("db:drugbank-id", NS):
        if dbid.get("primary") == "true":
            return get_text(dbid)
    first = drug.find("db:drugbank-id", NS)
    return get_text(first)

def get_smiles(drug):
    paths = [
        "db:calculated-properties/db:property",
        "db:experimental-properties/db:property"
    ]
    for path in paths:
        for prop in drug.findall(path, NS):
            kind = get_text(prop.find("db:kind", NS))
            if kind and kind.upper() == "SMILES":
                val = get_text(prop.find("db:value", NS))
                if val:
                    return val
    return get_text(drug.find("db:smiles", NS))


In [3]:
def get_kegg_drug_id(drug):
    ex_ids = drug.find("db:external-identifiers", NS)
    if ex_ids is None:
        return None
    for ex in ex_ids.findall("db:external-identifier", NS):
        res = get_text(ex.find("db:resource", NS))
        if res and "kegg" in res.lower():
            return get_text(ex.find("db:identifier", NS))
    return None


In [5]:
def extract_targets(drug):
    block = drug.find("db:targets", NS)
    if block is None:
        return []

    out = []
    for t in block.findall("db:target", NS):
        organism = get_text(t.find("db:organism", NS)) or ""
        polys = t.findall("db:polypeptide", NS)

        # try KEGG protein ID
        kegg_protein = None
        ext = t.find("db:external-identifiers", NS)
        if ext is not None:
            for ex in ext.findall("db:external-identifier", NS):
                res = get_text(ex.find("db:resource", NS))
                if res and "kegg" in res.lower():
                    kegg_protein = get_text(ex.find("db:identifier", NS))

        for p in polys:
            uniprot = p.get("id")
            seq = get_text(p.find("db:amino-acid-sequence", NS)) or ""
            if uniprot:
                out.append((uniprot, seq, organism, kegg_protein))

    return out


In [7]:
xml_path = "../data/positive dti datasets/drugbank/drugbank beginning dataset/full_database.xml"
out_csv = "drugbank_dti_with_kegg.csv"

tree = ET.parse(xml_path)
root = tree.getroot()

with open(out_csv, "w", newline="", encoding="utf8") as f:
    w = csv.writer(f)
    w.writerow([
        "drugbank_id",
        "drug_name",
        "smiles",
        "kegg_drug_id",
        "protein_uniprot_id",
        "kegg_protein_id",
        "protein_sequence",
        "organism",
        "label"
    ])

    total_rows = 0

    for drug in root.findall("db:drug", NS):
        if drug.get("type", "").lower() != "small molecule":
            continue

        did = get_primary_id(drug)
        name = get_text(drug.find("db:name", NS)) or ""
        smiles = get_smiles(drug)
        kegg_drug = get_kegg_drug_id(drug)

        if not did or not smiles:
            continue

        targets = extract_targets(drug)
        for uni, seq, org, kegg_prot in targets:
            w.writerow([
                did,
                name,
                smiles,
                kegg_drug,
                uni,
                kegg_prot,
                seq,
                org,
                1
            ])
            total_rows += 1

print("Done. Total pairs:", total_rows)
print("Saved to", out_csv)


Done. Total pairs: 24525
Saved to drugbank_dti_with_kegg.csv
